# Export the multi-head model to ONNX

`view` is a Python string selecting a decoder head. Export one file per view;
choose the corresponding file when running ONNX Runtime inference.
Each file accepts float32 `image` tensors shaped `(batch, channels, 256, 256)`
(with the spatial size taken from `cfg_data.RESIZE_DIM`) and returns class
probabilities, matching `model(image, view=view)` including its softmax.
Only the batch dimension is dynamic. Apply the same image preprocessing as in PyTorch.

Two checkpoints are exported (see `checkpoints` below): `saved_models/CINE/last.ckpt` and
`saved_models/LGE/lge_epoch=499.ckpt`. The views of each checkpoint (and their number of classes)
are read from the checkpoint itself, so the LGE model also exports its `SAX_INFARCT` head.

Run the cells in order from the project root or the `notebooks` directory.
The notebook uses CPU for portable export and numerical verification.
If the export dependencies are missing, run `%pip install onnx onnxscript onnxruntime`
in the notebook kernel (use the existing `onnxruntime-gpu` instead of installing
`onnxruntime` if it is already present).

[PyTorch ONNX export documentation](https://docs.pytorch.org/docs/stable/onnx_export.html)
explains how non-tensor arguments are fixed at export time.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import onnx
import onnxruntime as ort
import torch
from hydra import compose, initialize_config_dir

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "config").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "segment2d").is_dir():
    raise FileNotFoundError("Run this notebook from the project root or notebooks directory.")

src_dir = PROJECT_ROOT / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
from segment2d import NATA_Net

config_dir = PROJECT_ROOT / "config"
saved_model_dir = PROJECT_ROOT / "saved_models"
with initialize_config_dir(version_base=None, config_dir=str(config_dir)):
    cfg_data = compose(config_name="data_config")
    cfg_train = compose(config_name="train_config")

In [ ]:
class ViewModel(torch.nn.Module):
    """Bind the view so the exported graph only needs an image tensor."""

    def __init__(self, model, view):
        super().__init__()
        self.model = model
        self.view = view

    def forward(self, image):
        return self.model(image, view=self.view)


# name -> checkpoint to export; the ONNX files are named `NATA-Net_<name>_<view>.onnx`
checkpoints = {
    "CINE": saved_model_dir / "CINE" / "last.ckpt",
    "LGE": saved_model_dir / "LGE" / "lge_epoch=499.ckpt",
}


def load_nata_net(checkpoint_path):
    """Build a NATA_Net whose heads match the checkpoint, load its weights and return (eager CPU model, {view: n_classes})."""
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    # Training saved a Lightning `model` containing a torch.compile `_orig_mod`.
    state_dict = {
        key.removeprefix("model.").removeprefix("_orig_mod."): value
        for key, value in checkpoint["state_dict"].items()
    }
    # the last conv of every head (`finalHeads.<view>.block.6`) has one output channel per class
    head_classes = {
        key.split(".")[1]: int(value.shape[0])
        for key, value in state_dict.items()
        if key.startswith("finalHeads.") and key.endswith(".block.6.weight")
    }
    model = NATA_Net(in_channels=int(cfg_train.INPUT_DIM_MODEL), head_classes=head_classes)
    model.load_state_dict(state_dict, strict=True)
    return model.cpu().float().eval(), head_classes

In [ ]:
# Export every view of every checkpoint.
output_dir = saved_model_dir / "onnx"
output_dir.mkdir(parents=True, exist_ok=True)
image_size = int(cfg_data.RESIZE_DIM)
in_channels = int(cfg_train.INPUT_DIM_MODEL)
generator = torch.Generator().manual_seed(42)
image = torch.randn(1, in_channels, image_size, image_size, generator=generator)
onnx_paths = {}

for name, checkpoint_path in checkpoints.items():
    model, head_classes = load_nata_net(checkpoint_path)
    print(f"Loaded {checkpoint_path} (views: {head_classes})")

    for view in head_classes:
        export_model = ViewModel(model, view).eval()
        onnx_path = output_dir / f"NATA-Net_{name}_{view}.onnx"
        with torch.inference_mode():
            torch.onnx.export(
                export_model,
                (image,),
                str(onnx_path),
                input_names=["image"],
                output_names=["probabilities"],
                opset_version=18,
                dynamo=True,
                dynamic_shapes={"image": {0: torch.export.Dim("batch", min=1)}},
                external_data=False,
            )
        onnx.checker.check_model(str(onnx_path))
        onnx_paths[(name, view)] = onnx_path
        print(f"Exported {name}/{view}: {onnx_path}")

## Verify predictions and the dynamic batch dimension

Compare every exported checkpoint/view with its PyTorch checkpoint at batch sizes 1 and 2.
Checks use synthetic inputs and `rtol=1e-1`, `atol=1e-1`.
The report includes maximum probability error and pixelwise segmentation agreement;
near-tied probabilities can produce slightly different argmax masks.
Run after export, or run the setup cells and set `output_dir = saved_model_dir / "onnx"`
to verify existing exports. This checks conversion accuracy, not clinical accuracy.

In [ ]:
# Verify every checkpoint/view against its matching ONNX file.
image_size = int(cfg_data.RESIZE_DIM)
in_channels = int(cfg_train.INPUT_DIM_MODEL)
verification_results = []
verify_generator = torch.Generator().manual_seed(123)

for name, checkpoint_path in checkpoints.items():
    reference_model, head_classes = load_nata_net(checkpoint_path)

    for verify_view, num_classes in head_classes.items():
        onnx_path = output_dir / f"NATA-Net_{name}_{verify_view}.onnx"
        onnx.checker.check_model(str(onnx_path))
        session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
        for batch_size in (1, 2):
            test_image = torch.randn(batch_size, in_channels, image_size, image_size, generator=verify_generator)
            with torch.inference_mode():
                expected = reference_model(test_image, view=verify_view).numpy()
            actual = session.run(["probabilities"], {"image": test_image.numpy()})[0]
            assert actual.shape == (batch_size, num_classes, image_size, image_size)
            assert np.isfinite(actual).all(), f"Non-finite output: {onnx_path.name}"
            np.testing.assert_allclose(
                actual, expected, rtol=1e-1, atol=1e-1, err_msg=f"{onnx_path.name}, batch={batch_size}"
            )
            max_error = float(np.max(np.abs(actual - expected)))
            mask_agreement = float(np.mean(actual.argmax(1) == expected.argmax(1)))
            verification_results.append({
                "model": onnx_path.name, "batch_size": batch_size,
                "max_absolute_error": max_error, "mask_agreement": mask_agreement,
            })
            print(
                f"PASS {onnx_path.name}, batch={batch_size}: "
                f"max error={max_error:.3g}, mask agreement={mask_agreement:.2%}"
            )
        del session
    del reference_model

print(f"Passed all {len(verification_results)} prediction comparisons.")

## Predict with ONNX Runtime

Select the checkpoint name and view explicitly. The helper accepts a PyTorch tensor
or NumPy array, converts it to float32, and checks its NCHW shape. Replace the synthetic
`image` below with your preprocessed batch, using the same preprocessing as PyTorch.
`y_pred` contains probabilities; `segmentation` contains class indices.

In [ ]:
# Choose the exported model independently of the export loop's last iteration.
predict_name = "CINE"  # "CINE" or "LGE"
view = "SAX"  # CINE: SAX, 2CH, 4CH | LGE: SAX_SCAR, 2CH, 4CH, RAS, SAX_INFARCT
onnx_path = output_dir / f"NATA-Net_{predict_name}_{view}.onnx"
session = ort.InferenceSession(
    str(onnx_path),
    providers=[
        "CUDAExecutionProvider",
        "CPUExecutionProvider",
    ],
)


def predict_onnx(image, session):
    """Predict probabilities and class masks from a preprocessed NCHW batch."""
    if isinstance(image, torch.Tensor):
        image = image.detach().cpu().float().numpy()
    image = np.ascontiguousarray(image, dtype=np.float32)
    input_info = session.get_inputs()[0]
    expected_shape = input_info.shape
    if image.ndim != 4 or image.shape[0] < 1:
        raise ValueError(f"Expected a nonempty NCHW batch; got {image.shape}.")
    for axis, expected_dim in enumerate(expected_shape):
        if isinstance(expected_dim, int) and image.shape[axis] != expected_dim:
            raise ValueError(f"Expected shape {expected_shape}; got {image.shape}.")
    probabilities = session.run(["probabilities"], {input_info.name: image})[0]
    return probabilities, probabilities.argmax(axis=1)


# Synthetic example: replace image with your preprocessed image batch.
image = torch.randn(1, int(cfg_train.INPUT_DIM_MODEL), int(cfg_data.RESIZE_DIM), int(cfg_data.RESIZE_DIM))
y_pred, segmentation = predict_onnx(image, session)
print(f"Model: {onnx_path.name}")
print("Probabilities:", y_pred.shape, "Segmentation:", segmentation.shape)